# Phase 3 Step 1a -- LLM model bake-off on the 58 gold reports

Picks which open-weight LLM runs the full 4,407-report pass (Step 1b) by measuring gold AUC
directly, rather than by picking a name off a plan.

**Uses `transformers`, not vLLM.** Two GPU sessions were lost to vLLM install failures
(vllm-project/vllm#43435: the precompiled wheel is cu13 regardless of `--torch-backend`).
vLLM was only ever there for prefix caching, which was only needed because the prompt used to
prepend the whole 12-definition rubric to every question. Sending just the one relevant criterion
made prompts ~3x shorter and removed the dependency entirely -- `transformers` ships in the Kaggle
image, so there is no install step to fail.

Rules compliance: report text never leaves this notebook and is never sent to a hosted API.

In [ ]:
# Mount the private src/knee dataset. GIT_SHA is baked in at push time since
# Kaggle kernels have no git context.
import glob, os, shutil, sys, time

GIT_SHA = 'uncommitted-phase3-step1'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU attached -- pick GPU T4 x2 before Save & Run All'
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, compute capability sm_{major}{minor}')
    # A P100 (sm_60) has been assigned before despite selecting T4x2. Fail fast
    # and loud here rather than deep inside a CUDA kernel launch.
    assert (major, minor) >= (7, 0), (
        f'GPU {i} is sm_{major}{minor}, too old for fp16 inference on this torch build. '
        'Kaggle likely assigned a P100 -- stop this run, pick GPU T4 x2 explicitly in the '
        'notebook editor accelerator settings, then Save Version -> Save & Run All (Commit).'
    )
print(f'{n_gpus} GPU(s) OK')

import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret found')
except Exception:
    print('No HF_TOKEN Kaggle Secret -- gated models (Gemma-3-12B-IT) will record as a load failure')

In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.reports import build_label_prompt, build_lexical_labels

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58, f'expected 58 gold studies, got {len(gold_df)}'
assert gold_df['Report'].notna().all(), 'a gold study is missing report text'

reports = gold_df['Report'].tolist()
study_uids = gold_df['StudyInstanceUID'].tolist()
y_true = gold_df[LABEL_COLUMNS].to_numpy(dtype=float)
print(f'{len(gold_df)} gold studies loaded, all with report text')

In [ ]:
# Step 0d's lexical-vs-gold anchor, recomputed here rather than hardcoded so it
# can't drift from the current lexical rules. This is the go/no-go bar the
# winning candidate has to clear.
from sklearn.metrics import roc_auc_score

lexical_df = build_lexical_labels(gold_df[['StudyInstanceUID', 'Report']])
lexical_df = lexical_df.set_index('StudyInstanceUID').loc[study_uids].reset_index()
y_pred_lexical = lexical_df[LABEL_COLUMNS].to_numpy(dtype=float)

lexical_aucs = []
for i in range(len(LABEL_COLUMNS)):
    # mask on BOTH sides: a lexical rule that found no match is NaN in the
    # prediction, not just possibly in the target
    mask = ~np.isnan(y_true[:, i]) & ~np.isnan(y_pred_lexical[:, i])
    if mask.sum() == 0 or len(np.unique(y_true[mask, i])) < 2:
        continue
    lexical_aucs.append(roc_auc_score(y_true[mask, i], y_pred_lexical[mask, i]))

LEXICAL_ANCHOR = float(np.mean(lexical_aucs))
print(f'lexical-vs-gold macro AUC anchor: {LEXICAL_ANCHOR:.4f} '
      f'(over the {len(lexical_aucs)} labels with any lexical coverage)')

In [ ]:
# Candidates in order of increasing risk, so the harness is proven on the
# cheapest model before anything larger is attempted.
#
# Gemma-3-12B-IT is included only when an HF_TOKEN is actually present: the
# CPU probe confirmed it 401s as a gated repo without one, and burning session
# time on a download that cannot succeed is exactly the waste this pass exists
# to stop. To enable it: accept the license on huggingface.co/google/gemma-3-12b-it,
# then add the token as a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets).
CANDIDATES = [
    {'name': 'Qwen3-4B-Instruct', 'repo': 'Qwen/Qwen3-4B-Instruct-2507'},
    {'name': 'Qwen3-8B',          'repo': 'Qwen/Qwen3-8B'},
]
if os.environ.get('HF_TOKEN'):
    CANDIDATES.append({'name': 'Gemma-3-12B-IT', 'repo': 'google/gemma-3-12b-it'})
    print('HF_TOKEN present -- Gemma-3-12B-IT included')
else:
    print('No HF_TOKEN -- skipping Gemma-3-12B-IT (the CPU probe confirmed it 401s without one)')

BATCH_SIZE = 8
FULL_CORPUS_SIZE = 4407
SESSION_BUDGET_HOURS = 4.0
RESULTS_CSV = '/kaggle/working/bakeoff_results.csv'

# One prompt per (study, label). Built once and reused for every candidate, so
# each is measured on identical input.
PROMPTS = [(si, li) for si in range(len(reports)) for li in range(len(LABEL_COLUMNS))]
print(f'{len(PROMPTS)} prompts per candidate '
      f'({len(reports)} studies x {len(LABEL_COLUMNS)} labels), batch size {BATCH_SIZE}')

In [ ]:
import gc
from types import SimpleNamespace
from transformers import AutoModelForCausalLM, AutoTokenizer

from knee.metrics import per_label_auc
from knee.reports import score_from_top_logprobs

_YES_FORMS = ('Yes', 'yes', ' Yes', ' yes', 'YES')
_NO_FORMS = ('No', 'no', ' No', ' no', 'NO')


def single_token_forms(tok, forms):
    """{token_id: surface_form} for the forms this tokenizer encodes as exactly
    one token. Multi-token forms are unusable: the score is read at a single
    answer position, so a form spanning two tokens has no logit there. Which
    forms qualify is genuinely tokenizer-dependent (a Llama tokenizer offered
    only ' Yes'/'YES', while both Qwen3 tokenizers offer all six), which is why
    several are tried and the scorer takes the max rather than assuming one."""
    out = {}
    for form in forms:
        ids = tok.encode(form, add_special_tokens=False)
        if len(ids) == 1:
            out[ids[0]] = form
    return out


def render(tok, prompt):
    """Apply the chat template so the model's first generated token is the
    answer. enable_thinking=False is REQUIRED for Qwen3-8B: verified locally
    that its default template leaves the assistant turn open, so the first
    token would open a <think> block and every answer would score NaN. The
    4B-Instruct variant ignores the flag, and Gemma's template rejects it
    outright -- hence try/except rather than an assumption either way."""
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def score_batch(model, tok, rendered, yes_ids, no_ids):
    """One forward pass over a batch of already-rendered prompts; returns the
    per-prompt {token_id: Logprob-like} mapping score_from_top_logprobs takes.
    No generation loop: the answer is a single token, so its logits are just
    the last position's. Left padding puts that last real token at index -1 for
    every row -- validated locally by checking batched scores match one-at-a-
    time scores exactly."""
    # add_special_tokens=False: apply_chat_template already emitted whatever
    # special tokens the model expects, so letting the tokenizer add BOS
    # again would double it on any model that adds one (Llama-family does,
    # Qwen does not) -- a silent prompt corruption rather than an error.
    enc = tok(rendered, return_tensors='pt', padding=True, add_special_tokens=False,
              truncation=True, max_length=4096).to(model.device)
    logits = model(**enc).logits[:, -1, :].float()
    logprobs = torch.log_softmax(logits, dim=-1)
    return [{tid: SimpleNamespace(decoded_token=form, logprob=logprobs[row, tid].item())
             for tid, form in {**yes_ids, **no_ids}.items()}
            for row in range(logprobs.shape[0])]


def run_candidate(name, repo):
    print(f'=== {name} ({repo}) ===', flush=True)
    try:
        tok = AutoTokenizer.from_pretrained(repo, padding_side='left')
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        # torch_dtype (not dtype): deprecated in transformers 5.x but still
        # functional, and valid on 4.x too. Kaggle currently ships 5.0.0.
        model = AutoModelForCausalLM.from_pretrained(
            repo, torch_dtype=torch.float16, device_map='auto')
        model.eval()
    except Exception as e:
        print(f'  LOAD FAILED: {type(e).__name__}: {e}', flush=True)
        return {'candidate': name, 'repo': repo, 'fits': False,
                'stage_failed': 'load', 'error': f'{type(e).__name__}: {e}'}

    yes_ids = single_token_forms(tok, _YES_FORMS)
    no_ids = single_token_forms(tok, _NO_FORMS)
    print(f'  yes tokens: {list(yes_ids.values())} | no tokens: {list(no_ids.values())}')
    if not yes_ids or not no_ids:
        del model; gc.collect(); torch.cuda.empty_cache()
        return {'candidate': name, 'repo': repo, 'fits': False, 'stage_failed': 'tokenizer',
                'error': 'no single-token Yes/No form -- nothing to score at the answer position'}

    # Render once, then order by token length before batching. The CPU probe
    # measured 389 tokens for a median report against 1287 for the longest, so
    # batching in corpus order would pad most batches to roughly 3x the tokens
    # they need. Sorting by length keeps each batch near-uniform; results are
    # scattered back by original index, so ordering never reaches the output.
    rendered = {(si, li): render(tok, build_label_prompt(reports[si], LABEL_COLUMNS[li]))
                for si, li in PROMPTS}
    lengths = {k: len(tok(v, add_special_tokens=False)['input_ids'])
               for k, v in rendered.items()}
    order = sorted(PROMPTS, key=lambda k: lengths[k])
    lens = list(lengths.values())
    print(f'  prompt tokens: min {min(lens)}, median {sorted(lens)[len(lens) // 2]}, '
          f'max {max(lens)}')

    y_pred = np.full((len(reports), len(LABEL_COLUMNS)), np.nan)
    weights = np.zeros_like(y_pred)
    t0 = time.time()
    try:
        for start in range(0, len(order), BATCH_SIZE):
            chunk = order[start:start + BATCH_SIZE]
            for key, mapping in zip(chunk, score_batch(
                    model, tok, [rendered[k] for k in chunk], yes_ids, no_ids)):
                score, weight = score_from_top_logprobs(mapping)
                si, li = key
                y_pred[si, li] = np.nan if score is None else score
                weights[si, li] = weight
            if start % (BATCH_SIZE * 20) == 0:
                print(f'    {start + len(chunk)}/{len(order)} ({time.time() - t0:.0f}s)',
                      flush=True)
    except Exception as e:
        print(f'  GENERATE FAILED: {type(e).__name__}: {e}', flush=True)
        del model; gc.collect(); torch.cuda.empty_cache()
        return {'candidate': name, 'repo': repo, 'fits': False,
                'stage_failed': 'generate', 'error': f'{type(e).__name__}: {e}'}
    elapsed = time.time() - t0

    aucs = per_label_auc(y_true, y_pred)
    macro = float(np.nanmean(aucs))
    answered = int((~np.isnan(y_pred)).sum())
    reports_per_sec = len(reports) / elapsed
    projected_hours = FULL_CORPUS_SIZE / reports_per_sec / 3600

    print(f'  gold macro AUC={macro:.4f} | answered {answered}/{y_pred.size} '
          f'({100 * answered / y_pred.size:.1f}%) | {elapsed:.0f}s for {len(reports)} reports '
          f'-> projected full corpus {projected_hours:.2f}h', flush=True)

    np.save(f'/kaggle/working/scores_{name}.npy', y_pred)
    np.save(f'/kaggle/working/weights_{name}.npy', weights)

    del model; gc.collect(); torch.cuda.empty_cache()
    return {
        'candidate': name, 'repo': repo, 'fits': projected_hours <= SESSION_BUDGET_HOURS,
        'stage_failed': None, 'error': None, 'macro_auc': macro,
        'answered': answered, 'n_total': int(y_pred.size),
        'answer_rate': answered / y_pred.size,
        'median_prompt_tokens': sorted(lens)[len(lens) // 2], 'max_prompt_tokens': max(lens),
        'elapsed_s': elapsed, 'reports_per_sec': reports_per_sec,
        'projected_full_corpus_hours': projected_hours,
        **{f'auc_{l}': a for l, a in zip(LABEL_COLUMNS, aucs.tolist())},
    }

In [ ]:
# Append after EACH candidate, not once at the end -- a crash or timeout on a
# later candidate must not throw away the earlier ones. That is the concrete
# lesson from the two sessions already lost.
results = []
for cand in CANDIDATES:
    results.append(run_candidate(cand['name'], cand['repo']))
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    print(f'  -> {RESULTS_CSV} updated ({len(results)} candidate(s) recorded)\n', flush=True)

In [ ]:
results_df = pd.DataFrame(results)
cols = [c for c in ['candidate', 'fits', 'macro_auc', 'answer_rate', 'elapsed_s',
                    'projected_full_corpus_hours', 'stage_failed'] if c in results_df]
print(results_df[cols].to_string(index=False))

eligible = [r for r in results if r.get('fits')]
if not eligible:
    print('\nNO CANDIDATE FITS -- see stage_failed/error above. Do not proceed to Step 1b.')
else:
    winner = max(eligible, key=lambda r: r['macro_auc'])
    NEAR_TIE = 0.02
    close = [r for r in eligible if winner['macro_auc'] - r['macro_auc'] <= NEAR_TIE]
    if len(close) > 1:
        winner = max(close, key=lambda r: r['reports_per_sec'])
        print(f'\n{len(close)} candidates within {NEAR_TIE} AUC -- taking the fastest '
              f'({winner["candidate"]}); a bigger model has to earn its cost.')

    print(f'\nWINNER: {winner["candidate"]} ({winner["repo"]})')
    print(f'  gold macro AUC {winner["macro_auc"]:.4f} | answer rate '
          f'{100 * winner["answer_rate"]:.1f}% | projected full corpus '
          f'{winner["projected_full_corpus_hours"]:.2f}h')
    print('  per-label AUC:')
    for label in LABEL_COLUMNS:
        print(f'    {label:20s} {winner.get(f"auc_{label}")}')

    margin = winner['macro_auc'] - LEXICAL_ANCHOR
    print(f'\nGo/no-go vs lexical anchor ({LEXICAL_ANCHOR:.4f}): margin {margin:+.4f}')
    if margin <= 0:
        print('  DOES NOT CLEAR the anchor -- stop and reconsider before spending the full '
              '4,407-report budget on this generator.')
    elif winner['answer_rate'] < 0.9:
        print(f'  Clears the anchor, BUT only {100 * winner["answer_rate"]:.1f}% of questions got '
              'a usable Yes/No -- investigate the chat template before trusting this.')
    else:
        print('  Clears the anchor -- proceed to Step 1b with this candidate.')

    print('\nCopy the table above into NOTES.md as the bake-off record before starting Step 1b.')